In [1]:
import mindspore
print("MindSpore version:", mindspore.__version__)
print("Ready to build Micra H Sense AI!")

MindSpore version: 1.7.0
Ready to build Micra H Sense AI!


In [2]:
import pandas as pd

df = pd.read_csv('micra_h_sense_synthetic_data (500).csv')
print("Shape:", df.shape)
print("\nEmotion labels:")
print(df['emotion_label'].value_counts())

Shape: (4000, 23)

Emotion labels:
Fear_Panic      500
Sadness         500
Acute_Stress    500
Depression      500
Calm            500
Anxiety         500
Happiness       500
Anger           500
Name: emotion_label, dtype: int64


In [3]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Feature columns
FEATURE_COLS = [
    'cortisol_nmol_L', 'epinephrine_pg_mL', 'norepinephrine_pg_mL',
    'acth_pg_mL', 'testosterone_ng_dL', 'serotonin_ng_mL',
    'estradiol_pg_mL', 'progesterone_ng_mL', 'dopamine_pg_mL',
    'oxytocin_pg_mL', 'bdnf_ng_mL', 'gaba_nmol_mL',
    'prolactin_ng_mL', 'il6_pg_mL', 'insulin_uIU_mL',
    'leptin_ng_mL', 'vasopressin_pg_mL', 'melatonin_pg_mL',
    'dheas_ug_dL', 'ne_epi_ratio'
]

# Encode labels
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['emotion_label'])
print("Emotion classes:", list(le.classes_))

# Features and labels
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label_encoded'].values.astype(np.int32)

# Normalize
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Features         : {X_train.shape[1]}")

Emotion classes: ['Acute_Stress', 'Anger', 'Anxiety', 'Calm', 'Depression', 'Fear_Panic', 'Happiness', 'Sadness']
Training samples : 3200
Testing samples  : 800
Features         : 20


In [4]:
import mindspore
import mindspore.nn as nn
import mindspore.dataset as ds
from mindspore import Tensor, context

context.set_context(mode=context.GRAPH_MODE, device_target="CPU")

# Build dataset
BATCH_SIZE = 32

def make_dataset(X, y, shuffle=True):
    data = {"features": X, "label": y}
    dataset = ds.NumpySlicesDataset(data, shuffle=shuffle)
    dataset = dataset.batch(BATCH_SIZE)
    return dataset

train_dataset = make_dataset(X_train, y_train, shuffle=True)
test_dataset  = make_dataset(X_test,  y_test,  shuffle=False)

# Define the neural network
class HormoneEmotionNet(nn.Cell):
    def __init__(self):
        super(HormoneEmotionNet, self).__init__()
        self.network = nn.SequentialCell([
            nn.Dense(20, 128), nn.ReLU(),
            nn.Dense(128, 64), nn.ReLU(),
            nn.Dense(64, 32),  nn.ReLU(),
            nn.Dense(32, 8)
        ])

    def construct(self, x):
        return self.network(x)

model = HormoneEmotionNet()
print(model)
print("\nModel ready!")

HormoneEmotionNet<
  (network): SequentialCell<
    (0): Dense<input_channels=20, output_channels=128, has_bias=True>
    (1): ReLU<>
    (2): Dense<input_channels=128, output_channels=64, has_bias=True>
    (3): ReLU<>
    (4): Dense<input_channels=64, output_channels=32, has_bias=True>
    (5): ReLU<>
    (6): Dense<input_channels=32, output_channels=8, has_bias=True>
    >
  >

Model ready!


In [5]:
from mindspore.nn import SoftmaxCrossEntropyWithLogits
from mindspore import Tensor
import mindspore.ops as ops

loss_fn   = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
optimizer = nn.Adam(model.trainable_params(), learning_rate=0.001)

net_with_loss = nn.WithLossCell(model, loss_fn)
train_net     = nn.TrainOneStepCell(net_with_loss, optimizer)
train_net.set_train()

print("Loss function and optimizer ready!")

Loss function and optimizer ready!


In [6]:
EPOCHS = 50
best_acc = 0.0

print("Starting training...\n")

for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    steps = 0

    for batch in train_dataset.create_dict_iterator():
        features = Tensor(batch["features"])
        labels   = Tensor(batch["label"])
        loss = train_net(features, labels)
        total_loss += float(loss.asnumpy())
        steps += 1

    avg_loss = total_loss / steps

    # Evaluate accuracy on test set
    model.set_train(False)
    correct, total = 0, 0
    for batch in test_dataset.create_dict_iterator():
        features = Tensor(batch["features"])
        labels   = batch["label"].asnumpy()
        logits   = model(features)
        preds    = logits.asnumpy().argmax(axis=1)
        correct += (preds == labels).sum()
        total   += len(labels)
    test_acc = correct / total
    model.set_train(True)

    if test_acc > best_acc:
        best_acc = test_acc
        mindspore.save_checkpoint(model, "best_model.ckpt")

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {test_acc*100:.2f}% | Best: {best_acc*100:.2f}%")

print(f"\nTraining complete! Best accuracy: {best_acc*100:.2f}%")

Starting training...

Epoch   1/50 | Loss: 1.5365 | Accuracy: 61.00% | Best: 61.00%
Epoch  10/50 | Loss: 0.0115 | Accuracy: 99.62% | Best: 99.62%
Epoch  20/50 | Loss: 0.0099 | Accuracy: 99.50% | Best: 99.75%
Epoch  30/50 | Loss: 0.0003 | Accuracy: 99.50% | Best: 99.75%
Epoch  40/50 | Loss: 0.0001 | Accuracy: 99.50% | Best: 99.75%
Epoch  50/50 | Loss: 0.0001 | Accuracy: 99.50% | Best: 99.75%

Training complete! Best accuracy: 99.75%


In [7]:
# Load deployment test data
deploy_df = pd.read_csv('micra_h_sense_deployment_test.csv')
print("Deployment samples:", len(deploy_df))
print("Columns:", deploy_df.columns.tolist())

# Prepare features
X_deploy = deploy_df[FEATURE_COLS].values.astype(np.float32)
X_deploy = scaler.transform(X_deploy).astype(np.float32)
true_labels = deploy_df['emotion_label'].values

# Run predictions
model.set_train(False)
all_preds = []

for i in range(0, len(X_deploy), 32):
    batch = Tensor(X_deploy[i:i+32])
    logits = model(batch).asnumpy()
    preds = logits.argmax(axis=1)
    all_preds.extend(preds.tolist())

# Decode predictions back to emotion names
pred_labels = le.inverse_transform(all_preds)
true_encoded = le.transform(true_labels)

# Results
from sklearn.metrics import classification_report, accuracy_score

acc = accuracy_score(true_encoded, all_preds)
print(f"\nDeployment Test Accuracy: {acc*100:.2f}%")
print("\nFull Report:")
print(classification_report(true_encoded, all_preds, target_names=le.classes_))

Deployment samples: 200
Columns: ['sample_id', 'emotion_label', 'sex', 'cortisol_nmol_L', 'epinephrine_pg_mL', 'norepinephrine_pg_mL', 'acth_pg_mL', 'testosterone_ng_dL', 'serotonin_ng_mL', 'estradiol_pg_mL', 'progesterone_ng_mL', 'dopamine_pg_mL', 'oxytocin_pg_mL', 'bdnf_ng_mL', 'gaba_nmol_mL', 'prolactin_ng_mL', 'il6_pg_mL', 'insulin_uIU_mL', 'leptin_ng_mL', 'vasopressin_pg_mL', 'melatonin_pg_mL', 'dheas_ug_dL', 'ne_epi_ratio']

Deployment Test Accuracy: 96.00%

Full Report:
              precision    recall  f1-score   support

Acute_Stress       0.92      1.00      0.96        23
       Anger       0.96      1.00      0.98        24
     Anxiety       0.92      1.00      0.96        23
        Calm       0.92      1.00      0.96        23
  Depression       1.00      0.96      0.98        26
  Fear_Panic       1.00      0.96      0.98        26
   Happiness       0.96      0.89      0.92        27
     Sadness       1.00      0.89      0.94        28

    accuracy                  